# Store Sales - EDA
Exploratory analysis for the Kaggle *Store Sales - Time Series Forecasting* competition.

Put the CSVs in `data/raw/` (or set the `STORE_SALES_DATA` environment variable) before running.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data import load_datasets
from src.config import DATA_DIR

ds = load_datasets(DATA_DIR)
train, test, stores, oil, hol, trans = (ds[k] for k in ["train", "test", "stores", "oil", "holidays", "transactions"])
print(train.shape, test.shape)
train.head()

## 1. Shape of the data

In [ ]:
print("Train :", train.date.min().date(), "->", train.date.max().date())
print("Test  :", test.date.min().date(), "->", test.date.max().date(), f"({test.date.nunique()} days)")
print("Stores:", train.store_nbr.nunique(), "| Families:", train.family.nunique())
print("Zero-sales share:", round((train.sales == 0).mean(), 3))
train.isna().sum()

## 2. Total daily sales
Note the upward trend, yearly seasonality, the dip on 1 January (stores closed) and the missing 25 December.

In [ ]:
daily = train.groupby("date").sales.sum()
ax = daily.plot(figsize=(12, 4), title="Total daily sales")
ax.axvline(pd.Timestamp("2016-04-16"), color="red", ls="--", label="Ecuador earthquake")
ax.legend(); plt.show()

## 3. Sales by product family

In [ ]:
fam = train.groupby("family").sales.sum().sort_values()
fam.plot.barh(figsize=(8, 8), title="Total sales by family"); plt.show()

## 4. Weekly and payday patterns

In [ ]:
t = train.assign(dow=train.date.dt.dayofweek, dom=train.date.dt.day)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
t.groupby("dow").sales.mean().plot.bar(ax=ax[0], title="Mean sales by day of week (0=Mon)")
t.groupby("dom").sales.mean().plot(ax=ax[1], title="Mean sales by day of month (paydays: 15th & month end)")
plt.show()

## 5. Promotions

In [ ]:
p = train[train.date >= "2015-01-01"]
p.groupby("onpromotion").sales.mean().head(30).plot(figsize=(10, 3.5), title="Mean sales vs number of items on promotion")
plt.show()

## 6. Oil price

In [ ]:
oil.set_index("date").dcoilwtico.plot(figsize=(12, 3), title="Daily oil price")
plt.show()

## 7. Transactions vs sales

In [ ]:
m = daily.rename("sales").reset_index().merge(trans.groupby("date").transactions.sum().reset_index(), on="date")
print("Correlation sales vs transactions:", round(m[["sales", "transactions"]].corr().iloc[0, 1], 3))

## 8. Holidays
A holiday with `transferred=True` is effectively a normal day; the `Transfer` row is the day it was celebrated.

In [ ]:
hol.groupby(["type", "locale"]).size().unstack(fill_value=0)

## 9. Stores

In [ ]:
stores.groupby(["type"]).size().plot.bar(title="Stores by type"); plt.show()
stores.head()